# Stage 3: DRT and Zarc Fitting

Tikhonov DRT on every selected spectrum, then fits R0 + Zarc1 + ... + ZarcN.

**Reads:** `{sample_id}/Results/{condition}/stage2_kk.xlsx` · `{sample_id}/ISM validation/`
**Writes:** `{sample_id}/Results/{condition}/stage3_drt.xlsx` · `stage3_fit.xlsx`

Set `FOCUS_T` to iterate on one temperature at a time (~10 s vs ~9 min full run).
The export is merge-aware: rows for other temperatures are preserved when `FOCUS_T` is set.

**Process identification:** the pipeline assigns no process name automatically.
Use C_eff magnitude and Arrhenius behaviour to identify each peak.
Starting-point thresholds from Vendrell & West 2018 (YSZ); verify empirically for your system:

| C_eff | Typical process |
|-------|-----------------|
| < 10⁻¹¹ F | bulk (grain interior) |
| 10⁻¹¹ to 10⁻⁸ F | grain boundary |
| 10⁻⁸ to 10⁻⁶ F | near-electrode |
| > 10⁻⁶ F | electrode |

## Quick links
- [Configuration](#configuration): sample_id, L_m, D_m, condition_filter, DRT and Zarc parameters
- [Condition selector](#condition-selector): FOCUS toggle for single temperature processing
- [Step 1: DRT computation](#step-1-drt-computation-and-peak-detection): heavy step, use FOCUS_T to iterate
- [Step 2: Zarc circuit fitting](#step-2-zarc-circuit-fitting): fast re-fit, no DRT recompute
- [Step 3: Arrhenius validation and export](#step-3-arrhenius-validation)

## Configuration

In [ ]:
import json
from pathlib import Path

_cfg      = json.loads(Path("session.json").read_text()) if Path("session.json").exists() else {}
sample_id = _cfg.get("sample_id") or input("Sample folder name: ").strip()

def _read_mm(prompt_label: str, stored_m) -> float:
    """Ask for a value in mm; accept comma or dot as decimal separator."""
    stored_mm = f"{stored_m * 1e3:.3f} mm" if stored_m is not None else "not set"
    raw = input(f"{prompt_label} [mm] (stored: {stored_mm}): ").strip().replace(",", ".")
    if raw:
        return float(raw) * 1e-3
    return stored_m

L_m = _read_mm("Thickness L", _cfg.get("L_m"))
D_m = _read_mm("Diameter  D", _cfg.get("D_m"))

_cfg["sample_id"] = sample_id
_cfg["L_m"]       = L_m
_cfg["D_m"]       = D_m
Path("session.json").write_text(json.dumps(_cfg, indent=2))

# conditions saved in stage 0; leave empty to process all
_saved           = _cfg.get("conditions", [])
condition_filter = _saved

FOCUS_T         = None   # None = all T, integer = only that T (e.g. 600)
FOCUS_CONDITION = None   # None = all conditions
SKIP_EXISTING   = False  # True = skip DRT compute if stage3_drt.xlsx already exists

# DRT parameters (calibrated on this dataset; change only to test literature defaults)
DRT_CV_TYPE    = "custom"     # 'mGCV', 'GCV', 'LC', 'custom'
DRT_DER        = "2nd order"  # '1st order' or '2nd order'
DRT_COEFF      = 0.5          # Gaussian RBF FWHM coefficient
DRT_REG_PARAM  = 4e-5         # lambda; only active when DRT_CV_TYPE='custom'
                               # higher = smoother DRT (fewer/broader peaks)

# Peak detection
PEAK_MIN_PROM_DECADES = 0.01  # log-prominence threshold; None = absolute-height mode only
PEAK_HEIGHT_FRAC      = 0.02  # absolute floor as fraction of gamma_max
PEAK_MIN_DIST         = 5

# Force N peaks per (condition, T); use when DRT picks spurious boundary peaks
# Example: N_PEAKS_OVERRIDE = {"condition_name": {400: 3, 450: 4}}
N_PEAKS_OVERRIDE = {}

# Zarc fitting parameters (calibrated on this dataset)
ZARC_INCLUDE_R0  = True    # False = RelaxIS-style pure-Zarc, no R0
ZARC_R0_MAX      = 200     # [Ohm] upper bound for R0; None = legacy guess-based bounds
ZARC_R_DEC       = 1.5     # R bounds span +-R_DEC decades around DRT area
ZARC_TAU_DEC     = 1.5     # tau bounds span +-TAU_DEC decades around DRT peak
ZARC_ALPHA_INIT  = 0.8     # initial CPE exponent for every Zarc element

# Per-condition/T overrides; priority: T-specific > condition > global
# Example: ZARC_OVERRIDE = {"condition": {"R_dec": 2.0, 400: {"tau_dec": 1.0}}}
ZARC_OVERRIDE = {}

# Fix specific Zarc parameters to constants (RelaxIS-style "tick to fix")
# Example: ZARC_FIX_PARAMS = {"condition": {400: {"R": [None, 200.0, None]}}}
ZARC_FIX_PARAMS = {}

## Condition selector

Toggle FOCUS to process one temperature at a time. Leave FOCUS off to process all conditions.

In [ ]:
# FOCUS selector: pick a condition and/or temperature by clicking (no config edit).
from pipeline.interactive import discover_conditions, make_focus_panel


def _apply_focus_nb03(cond, T):
    global FOCUS_CONDITION, FOCUS_T
    FOCUS_CONDITION = cond
    FOCUS_T = T


make_focus_panel(
    conditions = discover_conditions(_nbdir / sample_id, require="stage2_kk.xlsx"),
    temps      = [600, 575, 550, 525, 500, 475, 450, 425, 400],
    set_focus  = _apply_focus_nb03,
    init_cond  = FOCUS_CONDITION,
    init_T     = FOCUS_T,
)


## Import

In [ ]:
%matplotlib inline

import sys
import gc
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import stats as _stats

NOTEBOOK_DIR = Path().resolve()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from pipeline.ingest import load_ism
from pipeline.drt import clip_spectrum, compute_drt, find_drt_peaks
from pipeline.fitting import fit_zarc, fit_to_rows, conductivity, build_circuit_string
from pipeline.plots import (
    apply_pub_style, COLOR_MAP,
    plot_drt_stacked, plot_tau_arrhenius_consistency, plot_tau_tracks,
    plot_ceff_magnitude,
)

apply_pub_style()

if L_m is None or D_m is None:
    raise ValueError(
        "L_m and D_m must be set in the Configuration cell "
        "(sample thickness and diameter are required for conductivity calculations)."
    )

sample_dir   = NOTEBOOK_DIR / sample_id
A_m2         = np.pi * (D_m / 2) ** 2
results_base = sample_dir / "Results"
all_conditions = sorted([
    d.name for d in results_base.iterdir()
    if d.is_dir() and (d / "stage2_kk.xlsx").exists()
])
conditions = (
    [c for c in all_conditions if c in condition_filter]
    if condition_filter else all_conditions
)
if FOCUS_CONDITION is not None:
    conditions = [c for c in conditions if c == FOCUS_CONDITION]
    print(f"FOCUS_CONDITION active: {FOCUS_CONDITION}")

print(f"Sample    : {sample_id}")
print(f"Geometry  : L = {L_m*1e3:.3f} mm  |  D = {D_m*1e3:.3f} mm  |  A = {A_m2*1e6:.3f} mm²")
print(f"Conditions ({len(conditions)}):")
for c in conditions:
    print(f"  {c}")
if FOCUS_T is not None:
    print(f"\nFOCUS_T = {FOCUS_T} C: only this T will be processed")


def _resolve_zarc_params(condition: str, T_int: int) -> dict:
    cond_ov = ZARC_OVERRIDE.get(condition, {})
    t_ov    = cond_ov.get(T_int, {})
    return {
        "R_dec":      t_ov.get("R_dec",      cond_ov.get("R_dec",      ZARC_R_DEC)),
        "tau_dec":    t_ov.get("tau_dec",    cond_ov.get("tau_dec",    ZARC_TAU_DEC)),
        "alpha_init": t_ov.get("alpha_init", cond_ov.get("alpha_init", ZARC_ALPHA_INIT)),
    }

## Step 1: DRT computation and peak detection

Computes DRT for each (condition, T), detects peaks, and plots gamma(tau).
Use `FOCUS_T` to process one temperature at a time (~10 s). Re-running only Step 2 (Zarc fit) is ~0.5 s with no DRT recompute.
Spurious peaks: add the T to `N_PEAKS_OVERRIDE` in config and re-run this cell.

In [ ]:
# _drt_results[condition][T_nom] = {entry, peaks, freq, Z_re, Z_im, lv, fname, pO2}
# Persists in memory : the fitting cell (Step 2) reads this dict without recomputing DRT.
_drt_results = {}

for condition in conditions:
    print(f"\n{'─'*64}\nCondition: {condition}\n{'─'*64}", flush=True)

    xlsx_path = sample_dir / "Results" / condition / "stage2_kk.xlsx"
    df_sel    = pd.read_excel(xlsx_path, sheet_name="Selected")

    # SKIP_EXISTING: load from file if already computed
    if SKIP_EXISTING:
        drt_path = sample_dir / "Results" / condition / "stage3_drt.xlsx"
        if drt_path.exists():
            print(f"  [SKIP] Loading DRT from {drt_path.name}", flush=True)
            df_pk = pd.read_excel(drt_path, sheet_name="Peaks")
            df_sm = pd.read_excel(drt_path, sheet_name="Summary")
            _drt_results[condition] = {}
            for T_nom in sorted(df_pk["T_nominal"].unique(), reverse=True):
                T_nom  = int(T_nom)
                pkrows = df_pk[df_pk["T_nominal"] == T_nom]
                peaks  = pkrows[["peak_id","tau","gamma_peak","R_approx",
                                 "tau_left","tau_right"]].to_dict("records")
                sm_row = df_sm[df_sm["T_nominal"] == T_nom]
                lv     = float(sm_row["lambda"].values[0]) if (
                    not sm_row.empty and "lambda" in df_sm.columns) else float("nan")
                fname  = sm_row["file"].values[0] if not sm_row.empty else ""
                _drt_results[condition][T_nom] = {
                    "peaks": peaks, "entry": None,
                    "freq": None, "Z_re": None, "Z_im": None,
                    "lv": lv, "fname": fname, "pO2": None,
                }
            continue

    val_dir = sample_dir / "ISM validation" / condition
    _drt_results[condition] = {}

    for _, row in df_sel.sort_values("T_nominal", ascending=False).iterrows():
        T_nom = int(row["T_nominal"])
        if FOCUS_T is not None and T_nom != FOCUS_T:
            continue

        fname = row["file"]
        f_min = row["f_min_cut"] if pd.notna(row.get("f_min_cut")) else None
        f_max = row["f_max_cut"] if pd.notna(row.get("f_max_cut")) else None

        print(f"\n  T = {T_nom} °C  |  {fname}", flush=True)

        ism_path = val_dir / fname
        if not ism_path.exists():
            print(f"  [SKIP] Not found: {ism_path}")
            continue
        rec = load_ism(ism_path)
        freq, Z_re, Z_im = clip_spectrum(rec.freq, rec.Z_re, rec.Z_im, f_min, f_max)
        print(f"  Points: {len(freq)}  range: {freq.max():.1f} – {freq.min():.4f} Hz", flush=True)

        print(f"  DRT ({DRT_CV_TYPE}) ...", end=" ", flush=True)
        try:
            entry = compute_drt(freq, Z_re, Z_im,
                                cv_type=DRT_CV_TYPE, der_used=DRT_DER,
                                coeff=DRT_COEFF, reg_param=DRT_REG_PARAM,
                                suppress_output=True)
            lv = float(np.squeeze(entry.lambda_value))
            print(f"λ = {lv:.2e}", flush=True)
        except Exception as e:
            print(f"ERROR: {e}")
            continue

        # Peak detection + count override
        peaks = find_drt_peaks(entry, min_height_frac=PEAK_HEIGHT_FRAC,
                                min_distance=PEAK_MIN_DIST,
                                min_prom_decades=PEAK_MIN_PROM_DECADES)
        n_override = N_PEAKS_OVERRIDE.get(condition, {}).get(T_nom)
        if n_override is not None:
            by_height = sorted(peaks, key=lambda p: p["gamma_peak"], reverse=True)
            peaks = sorted(by_height[:n_override], key=lambda p: p["tau"])
            for k, p in enumerate(peaks):
                p["peak_id"] = k + 1
            print(f"  [OVERRIDE] N_peaks = {n_override}", flush=True)

        n_peaks = len(peaks)
        print(f"  Peaks: {n_peaks}", flush=True)
        for p in peaks:
            print(f"    #{p['peak_id']}: τ={p['tau']:.3e} s  "
                  f"R_drt={p['R_approx']:.2f} Ω  "
                  f"γ_peak={p['gamma_peak']:.3f} Ω", flush=True)

        # DRT plot with physical guide lines and adaptive peak annotations
        _color = COLOR_MAP.get(T_nom, "#555555")
        y_max  = entry.gamma.max() * 1.20 if entry.gamma.max() > 0 else 1.0

        fig, ax = plt.subplots(figsize=(8, 3.5))
        ax.semilogx(entry.out_tau_vec, entry.gamma, "-", lw=1.5, color=_color)
        ax.fill_between(entry.out_tau_vec, 0, entry.gamma, alpha=0.12, color=_color)

        _ann_y = []
        for p in sorted(peaks, key=lambda p: p["gamma_peak"], reverse=True):
            _y_cand = p["gamma_peak"] + y_max * 0.09
            for _yp in _ann_y:
                if abs(_y_cand - _yp) < y_max * 0.14:
                    _y_cand = _yp + y_max * 0.14
            _ann_y.append(_y_cand)
            ax.axvline(p["tau"], color="grey", ls=":", lw=0.9, alpha=0.5)
            ax.annotate(
                f"#{p['peak_id']}  τ={p['tau']:.1e} s",
                xy=(p["tau"], p["gamma_peak"]),
                xytext=(p["tau"], min(_y_cand, y_max * 0.90)),
                arrowprops=dict(arrowstyle="-", color="grey", lw=0.7),
                fontsize=8, ha="center", va="bottom",
            )

        ax.set_xlabel(r"$\tau$ [s]")
        ax.set_ylabel(r"$\gamma(\log\tau)$  [Ω]")
        ax.set_title(
            f"T = {T_nom} °C   |   λ = {lv:.2e}   |   "
            f"N = {n_peaks} peak{'s' if n_peaks != 1 else ''}",
        )
        ax.set_xlim(entry.out_tau_vec.min(), entry.out_tau_vec.max())
        ax.set_ylim(bottom=0, top=y_max)
        ax.grid(True, which="both", ls=":", alpha=0.35)
        plt.tight_layout()
        plt.show()
        plt.close(fig)

        _drt_results[condition][T_nom] = {
            "entry": entry, "peaks": peaks,
            "freq": freq, "Z_re": Z_re, "Z_im": Z_im,
            "lv": lv, "fname": fname, "pO2": row.get("pO2_mean"),
        }
        gc.collect()

if FOCUS_T is not None:
    print(f"\nFOCUS_T={FOCUS_T} active; re-run after adjusting N_PEAKS_OVERRIDE.")
else:
    n_done = sum(len(v) for v in _drt_results.values())
    print(f"\n{'─'*64}")
    print(f"DRT complete: {n_done} spectra computed.")
    print("Check plots. Spurious peaks → adjust N_PEAKS_OVERRIDE (DRT Config cell).")

### Step 1b: Interactive DRT (gamma(tau) + peaks with live sliders)

Pick a (condition, T) and drag the sliders to see gamma(tau) and the detected peaks update in place (recomputes the DRT, ~1 s). `reg lambda` sets smoothness (higher = fewer/broader peaks), `prom`/`height` control peak detection. Uses `cv_type="custom"` so `reg lambda` is always active. Diagnostic only: does not overwrite the batch results. Set the values you like in the config cell, or use the Zarc tuning panel below.

In [ ]:
# Interactive DRT explorer: recompute gamma(tau) + peaks for one (cond, T), live.
try:
    import ipywidgets as W
    from IPython.display import display as _display, clear_output as _clear
    _OK_DRTW = True
except Exception as _e:
    _OK_DRTW = False
    print(f"[INFO] interactive DRT needs ipywidgets ({_e}).")

if _OK_DRTW and _drt_results:
    _cs = [c for c, d in _drt_results.items() if d]
    if _cs:
        dwc = W.Dropdown(options=_cs, value=_cs[0], description="Cond:",
                         layout=W.Layout(width="420px"))
        dwT = W.Dropdown(options=sorted(_drt_results[_cs[0]], reverse=True),
                         description="T [°C]:", layout=W.Layout(width="160px"))
        s_reg = W.FloatLogSlider(value=DRT_REG_PARAM, base=10, min=-6, max=-2, step=0.1,
                                 description="reg λ", readout_format=".1e",
                                 tooltip="Tikhonov regularisation λ: higher = smoother DRT, fewer/broader peaks",
                                 continuous_update=False, layout=W.Layout(width="380px"))
        s_prom = W.FloatSlider(value=(PEAK_MIN_PROM_DECADES or 0.05), min=0.0, max=0.5, step=0.01,
                               description="prom", readout_format=".2f",
                               tooltip="Peak log-prominence (decades): minimum local rise to be counted as a peak",
                               continuous_update=False, layout=W.Layout(width="320px"))
        s_h = W.FloatSlider(value=PEAK_HEIGHT_FRAC, min=0.0, max=0.2, step=0.01,
                            description="height", readout_format=".2f",
                            tooltip="Absolute height floor as fraction of gamma_max : rejects noise",
                            continuous_update=False, layout=W.Layout(width="320px"))
        out_drtw = W.Output()

        def _refresh_T_drtw(*_):
            ts = sorted(_drt_results.get(dwc.value, {}), reverse=True)
            dwT.options = ts
            if ts and dwT.value not in ts:
                dwT.value = ts[0]
        dwc.observe(_refresh_T_drtw, names="value")

        def _redraw_drtw(*_):
            with out_drtw:
                _clear(wait=True)
                drt = _drt_results.get(dwc.value, {}).get(int(dwT.value))
                if not drt or drt.get("freq") is None:
                    print("No cached freq/Z for this (cond, T); run Step 1 without SKIP_EXISTING.")
                    return
                entry = compute_drt(drt["freq"], drt["Z_re"], drt["Z_im"],
                                    cv_type="custom", der_used=DRT_DER,
                                    coeff=DRT_COEFF, reg_param=float(s_reg.value))
                prom = float(s_prom.value) if s_prom.value > 0 else None
                peaks = find_drt_peaks(entry, min_height_frac=float(s_h.value),
                                       min_distance=PEAK_MIN_DIST, min_prom_decades=prom)
                fig, ax = plt.subplots(figsize=(6, 4), dpi=150)
                ax.plot(entry.out_tau_vec, entry.gamma, color="#0066FF", lw=1.2)
                for p in peaks:
                    ax.axvline(p["tau"], color="crimson", ls="--", lw=0.8, alpha=0.7)
                ax.set_xscale("log")
                ax.set_xlabel(r"$\tau$ / s"); ax.set_ylabel(r"$\gamma(\tau)$ / $\Omega$")
                ax.grid(True, which="both", ls=":", alpha=0.3)
                plt.tight_layout(); plt.show()
                taus = ", ".join(f"{p['tau']:.1e}" for p in peaks)
                print(f"reg={s_reg.value:.2e}  prom={prom}  height={s_h.value}  ->  "
                      f"{len(peaks)} peaks  tau=[{taus}]")
        for _w in (dwc, dwT, s_reg, s_prom, s_h):
            _w.observe(_redraw_drtw, names="value")

        w_apply_drt = W.Button(description="📥 Apply to batch", button_style="warning",
                               layout=W.Layout(width="200px"),
                               tooltip="Copy these reg λ / prom / height into the batch DRT config (Step 1)")
        _apply_lbl = W.HTML()
        def _apply_drt_params(_b):
            global DRT_REG_PARAM, PEAK_MIN_PROM_DECADES, PEAK_HEIGHT_FRAC
            DRT_REG_PARAM = float(s_reg.value)
            PEAK_MIN_PROM_DECADES = float(s_prom.value) if s_prom.value > 0 else None
            PEAK_HEIGHT_FRAC = float(s_h.value)
            _apply_lbl.value = (
                f"<b style='color:#b36b00'>Applied</b>; DRT_REG_PARAM={DRT_REG_PARAM:.2e}, "
                f"PEAK_MIN_PROM_DECADES={PEAK_MIN_PROM_DECADES}, PEAK_HEIGHT_FRAC={PEAK_HEIGHT_FRAC}. "
                "Re-run Step 1 (batch DRT) to apply to all spectra.")
        w_apply_drt.on_click(_apply_drt_params)

        _redraw_drtw()
        _display(W.VBox([W.HBox([dwc, dwT]), s_reg, W.HBox([s_prom, s_h]),
                         W.HBox([w_apply_drt, _apply_lbl]), out_drtw]))
elif _OK_DRTW:
    print("[INFO] No DRT in memory; run Step 1 (DRT compute) first.")

## Adjust N peaks if needed

If a peak appears only at one temperature, does not track an Arrhenius line, or has inconsistent C_eff: add it to `N_PEAKS_OVERRIDE` in the config cell and re-run Step 1. Use `FOCUS_T` to iterate fast.

## Step 2: Zarc circuit fitting

Fits R0 + Zarc1 + ... + ZarcN using DRT peaks as starting points.
Reads `_drt_results` from memory; re-running this cell is ~0.5 s with no DRT recompute.
Title shown in red if rmse > 5%.

### Step 2b: Live tuning panel

Tune `N_PEAKS_OVERRIDE`, `ZARC_R_DEC`, `ZARC_TAU_DEC` per (condition, T) and re-fit a single spectrum (~0.5 s). DRT is reused from memory. Results persist to `stage3_fit.xlsx` only when you re-run the full Step 2 cell.

In [ ]:
# Live tuning panel: single-spectrum Zarc re-fit using DRT cached in _drt_results.
# Includes: ToggleButton (Include R0), slider (R0_max), preset buttons,
# status chips, peak checkbox list, state loading per (cond, T).
try:
    import ipywidgets as W
    from IPython.display import display as _display, clear_output as _clear, HTML as _HTML
    _HAS_WIDGETS_NB03 = True
except Exception as _exc:
    print(f"[INFO] ipywidgets not installed ({_exc}); tuning panel disabled.")
    _HAS_WIDGETS_NB03 = False


# Storage for per-(cond, T) peak deactivation choices in the widget.
# Key: (condition, T_int), value: list[bool] of length N (True = include peak).
_PEAK_KEEP_FLAGS: dict[tuple[str, int], list[bool]] = {}

# B1 fix: while True, programmatic widget value-sets must not trigger a re-fit
# (prevents the preset/condition-change observer cascade firing 2-4 fits).
_suspend_refit: bool = False


# Zarc presets: one-click parameter sets
_ZARC_PRESETS = {
    "Conservative (publication)": dict(
        ZARC_R_DEC=1.5, ZARC_TAU_DEC=1.5, ZARC_ALPHA_INIT=0.8,
        ZARC_INCLUDE_R0=True,  ZARC_R0_MAX=None,
        PEAK_MIN_PROM_DECADES=None, PEAK_HEIGHT_FRAC=0.10),
    "Standard (current optimum)": dict(
        ZARC_R_DEC=1.5, ZARC_TAU_DEC=1.5, ZARC_ALPHA_INIT=0.8,
        ZARC_INCLUDE_R0=True,  ZARC_R0_MAX=200,
        PEAK_MIN_PROM_DECADES=0.05, PEAK_HEIGHT_FRAC=0.02),
    "ceramic electrolyte-tight (max peaks)": dict(
        ZARC_R_DEC=2.0, ZARC_TAU_DEC=1.5, ZARC_ALPHA_INIT=0.7,
        ZARC_INCLUDE_R0=True,  ZARC_R0_MAX=200,
        PEAK_MIN_PROM_DECADES=0.03, PEAK_HEIGHT_FRAC=0.02),
    "RelaxIS-style (no R0)": dict(
        ZARC_R_DEC=1.5, ZARC_TAU_DEC=1.5, ZARC_ALPHA_INIT=0.8,
        ZARC_INCLUDE_R0=False, ZARC_R0_MAX=None,
        PEAK_MIN_PROM_DECADES=0.05, PEAK_HEIGHT_FRAC=0.02),
}


def _refit_one(condition: str, T_int: int, n_peaks: int | None,
               R_dec: float, tau_dec: float, alpha_init: float,
               include_r0: bool, r0_max: float | None,
               keep_flags: list[bool] | None = None,
               fix_params: dict | None = None) -> None:
    """Re-fit Zarc for a single (condition, T) using cached DRT.

    Uses the DRT entry cached in ``_drt_results``; never recomputes DRT.
    Also updates ``N_PEAKS_OVERRIDE`` and ``ZARC_OVERRIDE`` in memory so that
    re-running the main Step 2 cell preserves the choice.
    """
    drt = _drt_results.get(condition, {}).get(T_int)
    if drt is None:
        print(f"[WARN] no DRT for {condition} T={T_int} - run Step 1 first.")
        return
    peaks = list(drt["peaks"])
    # Apply keep_flags first (peak checkbox state)
    if keep_flags is not None and len(keep_flags) == len(peaks):
        peaks = [p for p, k in zip(peaks, keep_flags) if k]
        for j, p in enumerate(peaks):
            p["peak_id"] = j + 1
    # Then apply n_peaks override (keep N largest by gamma)
    if n_peaks is not None and n_peaks > 0 and peaks:
        by_h  = sorted(peaks, key=lambda p: p["gamma_peak"], reverse=True)
        peaks = sorted(by_h[:n_peaks], key=lambda p: p["tau"])
        for k, p in enumerate(peaks):
            p["peak_id"] = k + 1
        N_PEAKS_OVERRIDE.setdefault(condition, {})[T_int] = n_peaks
    ZARC_OVERRIDE.setdefault(condition, {}).setdefault(T_int, {}).update(
        {"R_dec": R_dec, "tau_dec": tau_dec, "alpha_init": alpha_init}
    )

    freq, Z_re, Z_im = drt["freq"], drt["Z_re"], drt["Z_im"]
    if freq is None:
        print("[WARN] DRT was loaded from SKIP_EXISTING (no freq/Z); cannot re-fit.")
        return

    fit = fit_zarc(freq, Z_re, Z_im, peaks, R0_guess=None,
                   R_dec=R_dec, tau_dec=tau_dec, alpha_init=alpha_init,
                   include_r0=include_r0, r0_max=r0_max,
                   fix_params=fix_params)
    print(f"  circuit: {fit['circuit_str']}")
    print(f"  N={len(peaks)}  status={'converged' if fit['converged'] else 'NOT CONVERGED'}"
          f"  rmse_rel={fit['rmse_rel']:.4f}  R0={fit['R0']:.4g} Ω")
    for i in range(len(peaks)):
        print(f"    Zarc{i+1}: R={fit['R'][i]:.4g} Ω  τ={fit['tau'][i]:.3e} s  "
              f"α={fit['alpha'][i]:.3f}  C_eff={fit['C_eff'][i]:.2e} F")

    Z_exp = Z_re - 1j * Z_im
    fig, ax = plt.subplots(figsize=(5.5, 5))
    color = COLOR_MAP.get(T_int, "#555555")
    ax.plot(Z_exp.real, -Z_exp.imag, "o", ms=4, color=color, label="data", zorder=3)
    ax.plot(fit["Z_fit"].real, -fit["Z_fit"].imag, "-", lw=1.5, color="tomato", label="fit")
    ax.set_xlabel(r"$Z'$ [Ω]"); ax.set_ylabel(r"$-Z''$ [Ω]")
    ax.set_aspect("equal", adjustable="datalim")
    ax.grid(True, ls=":", alpha=0.35)
    ax.legend(fontsize=9, frameon=False)
    ax.set_title(f"{condition}  |  T={T_int}°C  |  re-fit (panel)", fontsize=9)
    plt.tight_layout()
    plt.show()
    plt.close(fig)


def _status_chips_nb03() -> str:
    """Heuristic status counts based on DRT peak counts across cached spectra."""
    good = warn = bad = 0
    for cond, td in _drt_results.items():
        for T_int, drt in td.items():
            if drt is None:
                continue
            n = len(drt.get("peaks", []))
            if 3 <= n <= 5:
                good += 1
            elif n in (2, 6):
                warn += 1
            else:
                bad += 1
    return (f"<div style='font-size:13px; padding:4px 0'>"
            f"<span style='background:#d4edda; padding:3px 8px; border-radius:4px'>🟢 {good}</span>&nbsp; "
            f"<span style='background:#fff3cd; padding:3px 8px; border-radius:4px'>🟡 {warn}</span>&nbsp; "
            f"<span style='background:#f8d7da; padding:3px 8px; border-radius:4px'>🔴 {bad}</span>&nbsp;&nbsp; "
            f"<i>(N=3-5 ok, N=2/6 review, else suspicious)</i></div>")


if _HAS_WIDGETS_NB03 and _drt_results:
    _conds = [c for c, d in _drt_results.items() if d]
    if _conds:
        _c0  = _conds[0]
        _Ts  = sorted(_drt_results[_c0].keys(), reverse=True)
        _T0  = _Ts[0] if _Ts else 600

        w_c   = W.Dropdown(options=_conds, value=_c0, description="Cond:",
                           layout=W.Layout(width="420px"))
        w_T   = W.Dropdown(options=_Ts, value=_T0, description="T [°C]:",
                           layout=W.Layout(width="200px"))
        w_n   = W.IntText(value=0, description="N peaks",
                          tooltip="0 = use DRT-detected count",
                          layout=W.Layout(width="180px"))
        w_R   = W.FloatSlider(value=ZARC_R_DEC, min=0.5, max=3.0, step=0.1,
                              description="R_dec", readout_format=".1f",
                              tooltip="R bounds in log-decades around the DRT area (±) for the Zarc fit",
                              continuous_update=False, layout=W.Layout(width="300px"))
        w_tau = W.FloatSlider(value=ZARC_TAU_DEC, min=0.5, max=3.0, step=0.1,
                              description="τ_dec", readout_format=".1f",
                              tooltip="tau bounds in log-decades around the DRT peak position",
                              continuous_update=False, layout=W.Layout(width="300px"))
        w_a   = W.FloatSlider(value=ZARC_ALPHA_INIT, min=0.5, max=1.0, step=0.05,
                              description="α_init", readout_format=".2f",
                              tooltip="initial CPE exponent alpha for every Zarc element",
                              continuous_update=False, layout=W.Layout(width="300px"))

        w_r0_on  = W.ToggleButton(
            value=bool(ZARC_INCLUDE_R0),
            description=("Include R0" if ZARC_INCLUDE_R0 else "No R0"),
            button_style="info", layout=W.Layout(width="140px"))
        w_r0_max = W.FloatLogSlider(
            value=ZARC_R0_MAX if ZARC_R0_MAX else 200, base=10, min=0, max=4, step=0.1,
            description="R0_max Ω", readout_format=".0f",
            layout=W.Layout(width="320px"))

        w_go  = W.Button(description="↻ Re-fit", button_style="primary",
                         layout=W.Layout(width="140px"))
        w_preset = W.Dropdown(options=list(_ZARC_PRESETS), value="Standard (current optimum)",
                              description="Preset:", layout=W.Layout(width="380px"))
        w_apply  = W.Button(description="📥 Apply preset", button_style="warning",
                            layout=W.Layout(width="160px"))

        chips_n = W.HTML(value=_status_chips_nb03())
        peak_box = W.VBox([])
        out_n   = W.Output()

        def _load_state_into_widgets_nb03(*_):
            """Read overrides for current (cond, T) into widgets."""
            global _suspend_refit
            _suspend_refit = True
            cond, T = w_c.value, int(w_T.value)
            zov = ZARC_OVERRIDE.get(cond, {}).get(T, {})
            w_R.value   = float(zov.get("R_dec",      ZARC_R_DEC))
            w_tau.value = float(zov.get("tau_dec",    ZARC_TAU_DEC))
            w_a.value   = float(zov.get("alpha_init", ZARC_ALPHA_INIT))
            w_n.value   = int(N_PEAKS_OVERRIDE.get(cond, {}).get(T, 0) or 0)
            _suspend_refit = False
            drt = _drt_results.get(cond, {}).get(T)
            children = []
            if drt and drt.get("peaks"):
                peaks = drt["peaks"]
                key = (cond, T)
                flags = _PEAK_KEEP_FLAGS.setdefault(key, [True] * len(peaks))
                if len(flags) != len(peaks):
                    flags = [True] * len(peaks)
                    _PEAK_KEEP_FLAGS[key] = flags
                for i, p in enumerate(peaks):
                    f_eq = 1.0 / (2 * 3.14159265 * p["tau"])
                    cb = W.Checkbox(value=flags[i],
                                    description=f"Peak {p['peak_id']}  τ={p['tau']:.2e}s  "
                                                f"f={f_eq:.2e}Hz  γ={p['gamma_peak']:.0f}Ω",
                                    indent=False,
                                    layout=W.Layout(width="640px"))
                    def _make_cb(idx, _key=key):
                        def _f(change):
                            _PEAK_KEEP_FLAGS[_key][idx] = bool(change["new"])
                        return _f
                    cb.observe(_make_cb(i), names="value")
                    children.append(cb)
            peak_box.children = children

        def _refresh_Ts_nb03(*_):
            ts = sorted(_drt_results.get(w_c.value, {}).keys(), reverse=True)
            w_T.options = ts
            if ts and w_T.value not in ts:
                w_T.value = ts[0]
            _load_state_into_widgets_nb03()

        w_c.observe(_refresh_Ts_nb03, names="value")
        w_T.observe(_load_state_into_widgets_nb03, names="value")

        def _on_r0_toggle(change):
            global ZARC_INCLUDE_R0
            ZARC_INCLUDE_R0 = bool(change["new"])
            w_r0_on.description = "Include R0" if ZARC_INCLUDE_R0 else "No R0"
        w_r0_on.observe(_on_r0_toggle, names="value")

        def _on_refit(_btn):
            if _suspend_refit:
                return
            global ZARC_R0_MAX
            ZARC_R0_MAX = float(w_r0_max.value) if w_r0_on.value else None
            key = (w_c.value, int(w_T.value))
            keep_flags = _PEAK_KEEP_FLAGS.get(key)
            fix_params = (ZARC_FIX_PARAMS.get(w_c.value, {}) or {}).get(int(w_T.value))
            with out_n:
                _clear(wait=True)
                _refit_one(
                    w_c.value, int(w_T.value),
                    n_peaks    = int(w_n.value) if w_n.value > 0 else None,
                    R_dec      = float(w_R.value),
                    tau_dec    = float(w_tau.value),
                    alpha_init = float(w_a.value),
                    include_r0 = bool(w_r0_on.value),
                    r0_max     = float(w_r0_max.value) if w_r0_on.value else None,
                    keep_flags = keep_flags,
                    fix_params = fix_params,
                )
            chips_n.value = _status_chips_nb03()
        w_go.on_click(_on_refit)
        # Live re-fit on slider release (continuous_update=False avoids mid-drag fits)
        for _wsl in (w_R, w_tau, w_a, w_r0_max):
            _wsl.observe(lambda ch: _on_refit(None), names="value")

        def _on_apply_preset_nb03(_btn):
            global ZARC_R_DEC, ZARC_TAU_DEC, ZARC_ALPHA_INIT
            global ZARC_INCLUDE_R0, ZARC_R0_MAX
            global PEAK_MIN_PROM_DECADES, PEAK_HEIGHT_FRAC
            p = _ZARC_PRESETS[w_preset.value]
            ZARC_R_DEC      = p["ZARC_R_DEC"]
            ZARC_TAU_DEC    = p["ZARC_TAU_DEC"]
            ZARC_ALPHA_INIT = p["ZARC_ALPHA_INIT"]
            ZARC_INCLUDE_R0 = p["ZARC_INCLUDE_R0"]
            ZARC_R0_MAX     = p["ZARC_R0_MAX"]
            PEAK_MIN_PROM_DECADES = p["PEAK_MIN_PROM_DECADES"]
            PEAK_HEIGHT_FRAC      = p["PEAK_HEIGHT_FRAC"]
            global _suspend_refit
            _suspend_refit = True
            w_R.value = float(ZARC_R_DEC)
            w_tau.value = float(ZARC_TAU_DEC)
            w_a.value = float(ZARC_ALPHA_INIT)
            w_r0_on.value = bool(ZARC_INCLUDE_R0)
            if ZARC_R0_MAX:
                w_r0_max.value = float(ZARC_R0_MAX)
            _suspend_refit = False
            with out_n:
                _clear(wait=True)
                print(f"Loaded preset '{w_preset.value}':")
                for k, v in p.items():
                    print(f"  {k} = {v}")
                print("Re-run Step 1 (DRT) to apply peak-detection changes, then Step 2 (fit).")
        w_apply.on_click(_on_apply_preset_nb03)

        _load_state_into_widgets_nb03()
        _display(W.VBox([
            W.HBox([w_c, w_T]),
            W.HBox([w_n, w_R, w_tau, w_a]),
            W.HBox([w_r0_on, w_r0_max, w_go]),
            W.HBox([w_preset, w_apply]),
            W.HTML("<b>Peaks (uncheck to deactivate from next fit):</b>"),
            peak_box,
            chips_n, out_n,
        ]))
elif not _drt_results:
    print("[INFO] No DRT in memory - run Step 1 first.")


In [ ]:
# Reads from _drt_results (computed in Step 1); does NOT recompute DRT.
# To adjust the fit only, modify ZARC_OVERRIDE and re-run ONLY this cell.

all_results = {}

for condition in conditions:
    print(f"\n{'─'*64}\nCondition: {condition}\n{'─'*64}", flush=True)

    if condition not in _drt_results or not _drt_results[condition]:
        print("  [SKIP] No DRT in memory; run Step 1 first")
        all_results[condition] = {
            "drt_peaks": [], "drt_summary": [], "drt_spectra": [],
            "fit_peaks": [], "fit_summary": [],
        }
        continue

    xlsx_path = sample_dir / "Results" / condition / "stage2_kk.xlsx"
    df_sel    = pd.read_excel(xlsx_path, sheet_name="Selected")
    pO2_map   = {int(r["T_nominal"]): r.get("pO2_mean") for _, r in df_sel.iterrows()}

    cond_drt_peaks, cond_drt_summary, cond_drt_spectra = [], [], []
    cond_fit_peaks, cond_fit_summary = [], []

    for T_nom in sorted(_drt_results[condition].keys(), reverse=True):
        if FOCUS_T is not None and T_nom != FOCUS_T:
            continue

        drt   = _drt_results[condition][T_nom]
        peaks = drt["peaks"]
        entry = drt["entry"]
        freq  = drt["freq"]
        Z_re  = drt["Z_re"]
        Z_im  = drt["Z_im"]
        lv    = drt["lv"]
        fname = drt["fname"]
        pO2   = drt.get("pO2") or pO2_map.get(T_nom)
        T_K   = T_nom + 273.15

        print(f"\n  T = {T_nom} °C  |  {fname}", flush=True)

        if entry is not None:
            for tau_v, gamma_v in zip(entry.out_tau_vec, entry.gamma):
                cond_drt_spectra.append({
                    "condition": condition, "T_nominal": T_nom,
                    "tau": float(tau_v), "gamma": float(gamma_v),
                })
            cond_drt_summary.append({
                "condition": condition, "file": fname,
                "T_nominal": T_nom, "T_K": T_K, "pO2_mean": pO2,
                "N_peaks": len(peaks), "lambda": lv,
                "n_points_used": len(freq) if freq is not None else 0,
            })
        else:
            cond_drt_summary.append({
                "condition": condition, "file": fname,
                "T_nominal": T_nom, "T_K": T_K, "pO2_mean": pO2,
                "N_peaks": len(peaks), "lambda": lv,
            })

        for p in peaks:
            cond_drt_peaks.append({
                "condition": condition, "file": fname,
                "T_nominal": T_nom, "T_K": T_K, "pO2_mean": pO2, **p,
            })

        if not peaks:
            print("  [SKIP fit] No peaks.")
            continue
        if freq is None:
            print("  [SKIP fit] freq not available; remove SKIP_EXISTING and re-run Step 1")
            continue

        zarc_p  = _resolve_zarc_params(condition, T_nom)
        n_peaks = len(peaks)

        _ov_tag = ""
        if ZARC_OVERRIDE.get(condition):
            cov = ZARC_OVERRIDE[condition]
            if T_nom in cov or any(k in cov for k in ("R_dec", "tau_dec", "alpha_init")):
                _ov_tag = f"  [R_dec={zarc_p['R_dec']} τ_dec={zarc_p['tau_dec']}]"

        print(f"  Fitting {build_circuit_string(n_peaks)} ...{_ov_tag}", end=" ", flush=True)
        # R0_guess=None → improved HF intercept (10th percentile of positive Z_re in top 30% HF).
        # ZARC_R0_MAX clamps R0 bounds for ceramic electrolyte where true R0 is 1–100 Ω.
        # ZARC_INCLUDE_R0=False fits pure Zarc series (no R0, RelaxIS-style).
        # ZARC_FIX_PARAMS pins individual R_i / τ_i / α_i to constant values.
        _fix = (ZARC_FIX_PARAMS.get(condition, {}) or {}).get(T_nom)
        fit = fit_zarc(
            freq, Z_re, Z_im, peaks,
            R0_guess=None,
            R_dec=zarc_p["R_dec"],
            tau_dec=zarc_p["tau_dec"],
            alpha_init=zarc_p["alpha_init"],
            include_r0=ZARC_INCLUDE_R0,
            r0_max=ZARC_R0_MAX,
            fix_params=_fix,
        )
        status = "converged" if fit["converged"] else "NOT CONVERGED"
        print(f"{status}  rmse_rel={fit['rmse_rel']:.4f}", flush=True)
        print(f"    R0={fit['R0']:.4g} Ω", flush=True)
        for i in range(n_peaks):
            s    = conductivity(float(fit["R"][i]), L_m, D_m)
            C    = float(fit["C_eff"][i])
            if   C < 1e-11: proc = "bulk"
            elif C < 1e-8:  proc = "GB"
            elif C < 1e-6:  proc = "?"
            else:           proc = "electrode"
            print(f"    Zarc{i+1}: R={fit['R'][i]:.4g} Ω  τ={fit['tau'][i]:.3e} s  "
                  f"α={fit['alpha'][i]:.3f}  C_eff={C:.2e} F  [{proc}]", flush=True)

        if not fit["converged"] or fit["rmse_rel"] > 0.05:
            print(f"  Problematic fit T={T_nom}°C:")
            print(f"    N peaks: {n_peaks}")
            for i, p in enumerate(peaks):
                R_lo = p["R_approx"] / 10**zarc_p["R_dec"]
                R_hi = p["R_approx"] * 10**zarc_p["R_dec"]
                t_lo = p["tau"] / 10**zarc_p["tau_dec"]
                t_hi = p["tau"] * 10**zarc_p["tau_dec"]
                print(f"    Zarc{i+1}: R_DRT={p['R_approx']:.1f}Ω → bounds [{R_lo:.2f}, {R_hi:.2f}]")
                print(f"           τ_DRT={p['tau']:.3e}s → bounds [{t_lo:.2e}, {t_hi:.2e}]")
            print("    → Try: increase ZARC_R_DEC or fix N in N_PEAKS_OVERRIDE")

        Z_exp  = Z_re - 1j * Z_im
        Z_fit  = fit["Z_fit"]
        _color = COLOR_MAP.get(T_nom, "#555555")
        _tcol  = "tomato" if (not fit["converged"] or fit["rmse_rel"] > 0.05) else "black"

        fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
        fig.suptitle(
            f"T = {T_nom} °C   |   {build_circuit_string(n_peaks)}   |   "
            f"rmse = {fit['rmse_rel']:.4f}",
            fontsize=10, color=_tcol,
        )
        for ax in axes:
            ax.plot(Z_exp.real, -Z_exp.imag, "o", ms=4,
                    color=_color, label="data", zorder=3)
            ax.plot(Z_fit.real, -Z_fit.imag, "-", lw=1.5, color="tomato", label="fit")
            ax.set_xlabel(r"$Z'$ [Ω]")
            ax.set_ylabel(r"$-Z''$ [Ω]")
            ax.legend(fontsize=9, frameon=False)
            ax.grid(True, ls=":", alpha=0.35)
            ax.axhline(0, color="grey", lw=0.4)
        axes[0].set_title("Full range", fontsize=9)
        axes[0].set_aspect("equal", adjustable="datalim")

        R0f = fit["R0"]
        R1f = float(fit["R"][0])
        x_lo = max(0, np.percentile(Z_exp.real, 2))
        x_hi = R0f + R1f * 2
        y_hi = max(R1f * 0.6, 1.0)
        axes[1].set_xlim(x_lo, x_hi)
        axes[1].set_ylim(-y_hi * 0.1, y_hi)
        axes[1].set_aspect("equal", adjustable="datalim")
        axes[1].set_title("HF zoom (bulk arc)", fontsize=9)
        plt.tight_layout()
        plt.show()
        plt.close(fig)

        peak_rows, summary_row = fit_to_rows(
            fit, condition, fname,
            str(sample_dir / "ISM validation" / condition / fname),
            T_nom, pO2, L_m, D_m,
        )
        cond_fit_peaks.extend(peak_rows)
        cond_fit_summary.append(summary_row)
        gc.collect()

    all_results[condition] = {
        "drt_peaks":   cond_drt_peaks,
        "drt_summary": cond_drt_summary,
        "drt_spectra": cond_drt_spectra,
        "fit_peaks":   cond_fit_peaks,
        "fit_summary": cond_fit_summary,
    }

n_conv = sum(sum(1 for r in data["fit_summary"] if r.get("converged"))
             for data in all_results.values())
n_tot  = sum(len(data["fit_summary"]) for data in all_results.values())
if FOCUS_T is not None:
    print(f"\nFOCUS_T={FOCUS_T} active: only T={FOCUS_T}°C processed.")
else:
    print(f"\n{'─'*64}")
    print(f"Fitting complete: {n_conv}/{n_tot} converged.")
    print("Check Nyquist plots and the validation dashboard below.")

In [ ]:
# Validation dashboard: combines rmse and Arrhenius R² into one table

def _arrhenius_r2_by_peak(fit_peaks: list[dict]) -> dict[int, float]:
    if not fit_peaks:
        return {}
    df_loc = pd.DataFrame(fit_peaks)
    out = {}
    for pid in sorted(df_loc["peak_id"].unique()):
        sub  = df_loc[df_loc["peak_id"] == pid].dropna(subset=["tau_i", "T_nominal"])
        T_K  = sub["T_nominal"].values.astype(float) + 273.15
        ln_t = np.log(sub["tau_i"].values.astype(float))
        valid = np.isfinite(ln_t)
        if valid.sum() < 2:
            out[int(pid)] = float("nan")
            continue
        _, _, r, _, _ = _stats.linregress(1.0 / T_K[valid], ln_t[valid])
        out[int(pid)] = round(float(r) ** 2, 4)
    return out


for condition, data in all_results.items():
    if not data["fit_summary"]:
        continue

    r2_by_peak = _arrhenius_r2_by_peak(data["fit_peaks"])
    pid_max    = max(r2_by_peak.keys(), default=0)

    rows = []
    for row in sorted(data["fit_summary"], key=lambda r: r["T_nominal"], reverse=True):
        npk = int(row.get("N_peaks", 0))
        cvg = bool(row.get("converged", False))
        rms = float(row.get("rmse_rel", float("nan")))

        all_r2_ok = all(
            r2_by_peak.get(pid, 0.0) >= 0.97
            for pid in range(1, npk + 1)
        )
        if not cvg:
            status = "✗ not converged"
        elif rms >= 0.05:
            status = "⚠ rmse"
        elif not all_r2_ok:
            status = "⚠ Arrh."
        else:
            status = "✓"

        rec = {
            "T [°C]":   row["T_nominal"],
            "N":        npk,
            "R₀ [Ω]":  round(float(row.get("R0", float("nan"))), 3),
            "rmse":     round(rms, 4),
        }
        for pid in range(1, pid_max + 1):
            rec[f"R² pk{pid}"] = r2_by_peak.get(pid, float("nan"))
        rec["Status"] = status
        rows.append(rec)

    def _hl_dash(val):
        s = str(val)
        if "✗" in s: return "background-color: #f8d7da"
        if "⚠" in s: return "background-color: #fff3cd"
        if s == "✓":  return "background-color: #d4edda"
        return ""

    df_dash = pd.DataFrame(rows)
    r2_cols = [c for c in df_dash.columns if c.startswith("R²")]
    fmt = {"R₀ [Ω]": "{:.3f}", "rmse": "{:.4f}"}
    for c in r2_cols:
        fmt[c] = "{:.3f}"

    print(f"\nValidation dashboard: {condition}")
    display(
        df_dash.style
        .map(_hl_dash, subset=["Status"])
        .format(fmt, na_rep="n/a")
        .hide(axis="index")
        .set_caption(f"Fit + Arrhenius: {condition}")
    )

    n_bad = sum(1 for r in rows if "⚠" in r["Status"] or "✗" in r["Status"])
    if n_bad:
        print(f"  {n_bad} issue(s). Set FOCUS_T=X, adjust N_PEAKS_OVERRIDE or "
              "ZARC_OVERRIDE, re-run Step 1 or Step 2.")
    else:
        print("  All fits OK: converged, rmse < 5%, Arrhenius R² ≥ 0.97.")

## Adjust bounds if the fit fails

- `rmse` high with a clean DRT: widen bounds with `ZARC_R_DEC = 2.0` or `ZARC_TAU_DEC = 2.0`
- not converged: likely wrong N peaks; fix `N_PEAKS_OVERRIDE` and re-run Step 1
- only at low T (400-450 C): noisy signal or DRT boundary artifact; force N=2 in `N_PEAKS_OVERRIDE`

Use `FOCUS_T` + `ZARC_OVERRIDE` to fix one T, then re-run only Step 2.

## Fit summary and effective capacitance

Table A: fit status per T. `✓` = converged + rmse < 5% | `⚠ rmse high` = converged but inaccurate | `✗` = not converged.
Below each table: log10(C_eff) vs 1000/T per peak; read the physical process from capacitance magnitude. No process name is assigned automatically.

In [ ]:
# Fit summary per condition (Table A: fit status) + effective-capacitance plot.
# Process names are intentionally NOT assigned: read the process from the
# C_eff magnitude in the log10(C_eff) vs 1000/T plot below each table.
for condition, data in all_results.items():
    if not data["fit_summary"]:
        continue

    print(f"\n{'─'*64}")
    print(f"Condition: {condition}")

    rows_a = []
    for row in sorted(data["fit_summary"], key=lambda r: r["T_nominal"], reverse=True):
        if   row.get("converged") and row.get("rmse_rel", 1) < 0.05: ok = "✓"
        elif row.get("converged"):                                     ok = "⚠ rmse high"
        else:                                                          ok = "✗"
        rows_a.append({
            "T [°C]":    row["T_nominal"],
            "pO₂ [bar]": round(float(row.get("pO2_mean") or 0), 4),
            "file":      row.get("file", ""),
            "N":         row.get("N_peaks", ""),
            "R₀ [Ω]":   round(float(row.get("R0", float("nan"))), 3),
            "rmse":      round(float(row.get("rmse_rel", float("nan"))), 4),
            "OK":        ok,
        })

    def _hl_ok(val):
        if "✗" in str(val): return "background-color: #f8d7da"
        if "⚠" in str(val): return "background-color: #fff3cd"
        return ""

    display(
        pd.DataFrame(rows_a).style
        .map(_hl_ok, subset=["OK"])
        .format({"R₀ [Ω]": "{:.3f}", "rmse": "{:.4f}"})
        .hide(axis="index")
        .set_caption("Fit summary by temperature")
    )

    # Effective-capacitance magnitude; read the process from C_eff, no labels.
    if data["fit_peaks"]:
        df_peaks_c = pd.DataFrame(data["fit_peaks"])
        if {"peak_id", "T_nominal", "C_eff_i"}.issubset(df_peaks_c.columns):
            fig = plot_ceff_magnitude(df_peaks_c, condition=condition,
                                      save_dir=Path("."), save=False)
            plt.show()
            plt.close(fig)


## Step 3: Arrhenius validation

Every real physical process follows tau = tau0 * exp(Ea/kT), so ln(tau) vs 1000/T must be linear with R² >= 0.97.
R² < 0.97 means a likely spurious peak: fix `N_PEAKS_OVERRIDE` and re-run Step 1.
Ceramic electrolytes typically show Ea = 0.8-1.1 eV.

In [ ]:
# Stacked DRT: visual peak shift with T (normalized curve per temperature)
# Peaks must shift rightward (larger τ) as T decreases; thermally activated behaviour
# Diagnostic preview only; save=False, so no files are written here
# (the final saved figures are produced in 04_plots.ipynb).

for condition, data in all_results.items():
    if not data["drt_spectra"]:
        continue
    df_sp = pd.DataFrame(data["drt_spectra"])
    print(f"\nDRT Stacked: condition: {condition}")
    fig = plot_drt_stacked(
        df_sp,
        condition=condition,
        save_dir=Path("."),
        tau_max=1.0,
        offset_step=1.2,
        save=False,
    )
    plt.show()
    plt.close(fig)


In [ ]:
# Arrhenius τ validation: ln(τ) vs 1000/T per peak
# R² ≥ 0.97 → physically real process (thermally activated)
# R² < 0.97 → probable DRT artifact → fix N_PEAKS_OVERRIDE
# Diagnostic preview only; save=False, so no files are written here
# (the final saved figures are produced in 04_plots.ipynb).

for condition, data in all_results.items():
    if not data["fit_peaks"]:
        continue
    df_peaks_diag = pd.DataFrame(data["fit_peaks"])
    if not {"peak_id", "T_nominal", "tau_i"}.issubset(df_peaks_diag.columns):
        print(f"  [{condition}] missing columns for Arrhenius; skipping")
        continue

    print(f"\nArrhenius τ: condition: {condition}")
    fig = plot_tau_arrhenius_consistency(
        df_peaks_diag,
        condition=condition,
        save_dir=Path("."),
        r2_threshold=0.97,
        save=False,
    )
    plt.show()
    plt.close(fig)

    r2_map = _arrhenius_r2_by_peak(data["fit_peaks"])
    for pid, r2 in sorted(r2_map.items()):
        if np.isnan(r2):
            continue
        flag = "R²≥0.97 (OK)" if r2 >= 0.97 else "R²<0.97; possible spurious peak"
        print(f"  Peak {pid}: R² = {r2:.3f}  {flag}")


## Step 3b: tau-track diagnostic (cross-temperature peak alignment)

Plots `tau_i` (log) vs temperature for each `peak_id`, with bulk / GB / electrode tau bands, plus a peak-count-per-T table. A physically consistent process forms a smooth track; if peak count changes across temperatures, the same `peak_id` may map to different processes. Diagnostic only: nothing is saved. Use it to decide `N_PEAKS_OVERRIDE`.

In [ ]:
# tau-track diagnostic + peak-count-per-T table (diagnostic only, save=False).
for condition, data in all_results.items():
    if not data["fit_peaks"]:
        continue
    df_pk = pd.DataFrame(data["fit_peaks"])
    if not {"peak_id", "T_nominal", "tau_i"}.issubset(df_pk.columns):
        continue

    print(f"\ntau-tracks - condition: {condition}")
    fig = plot_tau_tracks(
        df_pk,
        condition=condition,
        save_dir=Path("."),
        save=False,
    )
    plt.show()
    plt.close(fig)

    counts = (df_pk.groupby("T_nominal")["peak_id"].nunique()
              .rename("n_peaks").sort_index(ascending=False))
    print(counts.to_string())
    n_set = sorted(counts.unique())
    if len(n_set) > 1:
        print(f"  [WARN] peak count varies across T ({n_set}) - peak_id may map to "
              "different processes; consider N_PEAKS_OVERRIDE to enforce consistency.")


## Step 3c: Export

Writes `stage3_drt.xlsx` (Peaks + Summary + DRT_Spectra) and `stage3_fit.xlsx` (Peaks + Summary) per condition.
When `FOCUS_T` is set, existing rows for other temperatures are preserved (merge-aware). `FOCUS_T = None` overwrites fully.

In [ ]:
from pipeline.utils import merge_sheet_by_T, build_metadata_sheet

# Build Metadata DataFrames (DRT + Zarc fixed parameters; applied to all conditions)
df_meta_drt = build_metadata_sheet(
    sample_id  = sample_id,
    stage_name = "stage3_drt",
    params = {
        "DRT_CV_TYPE":      DRT_CV_TYPE,
        "DRT_DER":          DRT_DER,
        "DRT_COEFF":        DRT_COEFF,
        "DRT_REG_PARAM":    DRT_REG_PARAM,
        "PEAK_HEIGHT_FRAC": PEAK_HEIGHT_FRAC,
        "PEAK_MIN_DIST":    PEAK_MIN_DIST,
        "reference":        "pyDRTtools (Wan, Ciucci et al., 2015); relaXIS manual §7.2",
    },
)
df_meta_fit = build_metadata_sheet(
    sample_id  = sample_id,
    stage_name = "stage3_fit",
    params = {
        "ZARC_R_DEC":      ZARC_R_DEC,
        "ZARC_TAU_DEC":    ZARC_TAU_DEC,
        "ZARC_ALPHA_INIT": ZARC_ALPHA_INIT,
        "circuit":         "R0 - Zarc_1 - ... - Zarc_N",
        "C_eff_formula":   "C_eff = Q^(1/alpha) * R^((1-alpha)/alpha)",
        "reference":       "Cole-Cole 1941; Boukamp 1986; Vendrell & West 2018",
    },
)

_export_mode = f"merged T={FOCUS_T}°C" if FOCUS_T is not None else "full overwrite"

for condition, data in all_results.items():
    results_dir = sample_dir / "Results" / condition
    results_dir.mkdir(parents=True, exist_ok=True)

    if data["drt_peaks"]:
        drt_path = results_dir / "stage3_drt.xlsx"
        df_pk = merge_sheet_by_T(drt_path, "Peaks",       pd.DataFrame(data["drt_peaks"]),   FOCUS_T)
        df_sm = merge_sheet_by_T(drt_path, "Summary",     pd.DataFrame(data["drt_summary"]), FOCUS_T)
        df_sp = merge_sheet_by_T(drt_path, "DRT_Spectra", pd.DataFrame(data["drt_spectra"]), FOCUS_T)
        with pd.ExcelWriter(drt_path, engine="openpyxl") as w:
            df_pk.to_excel(w,       sheet_name="Peaks",       index=False)
            df_sm.to_excel(w,       sheet_name="Summary",     index=False)
            df_sp.to_excel(w,       sheet_name="DRT_Spectra", index=False)
            df_meta_drt.to_excel(w, sheet_name="Metadata",    index=False)
        print(f"  [{condition}] stage3_drt.xlsx  ({len(df_sp)} spectral points)  [{_export_mode}]")

    if data["fit_peaks"]:
        fit_path = results_dir / "stage3_fit.xlsx"
        df_fp = merge_sheet_by_T(fit_path, "Peaks",   pd.DataFrame(data["fit_peaks"]),   FOCUS_T)
        df_fs = merge_sheet_by_T(fit_path, "Summary", pd.DataFrame(data["fit_summary"]), FOCUS_T)
        with pd.ExcelWriter(fit_path, engine="openpyxl") as w:
            df_fp.to_excel(w,       sheet_name="Peaks",    index=False)
            df_fs.to_excel(w,       sheet_name="Summary",  index=False)
            df_meta_fit.to_excel(w, sheet_name="Metadata", index=False)
        n_ok = int(df_fs["converged"].sum()) if "converged" in df_fs.columns else 0
        print(f"  [{condition}] stage3_fit.xlsx   ({n_ok}/{len(df_fs)} converged)  [{_export_mode}]")

print("\nExport complete.")
print("→ Next: 04_plots.ipynb")

**Next step:** run [stage4_plots.ipynb](stage4_plots.ipynb)